### Присоединяем

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
main = pd.read_excel('data/full_main.xlsx')
sanctions = pd.read_excel('data/sanctions_proxy.xlsx')
macro = pd.read_excel('data/macro_full.xlsx')
gscpi = pd.read_excel('data/gscpi_final.xlsx')
dist = pd.read_excel('data/dist_final.xlsx')

In [3]:
main = pd.merge(main, sanctions, on=['rep_date', 'country'], how='inner')
main = pd.merge(main, macro, on=['rep_date', 'country'], how='inner')
main = pd.merge(main, gscpi, on=['rep_date'], how='inner')
main = pd.merge(main, dist, on=['country'], how='inner')
main

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,distw
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355
1,2019-01-01,Albania,9019,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355
2,2019-01-01,Albania,9020,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355
3,2019-01-01,Albania,9021,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355
4,2019-01-01,Albania,9022,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355
...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-06-01,Uzbekistan,9022,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859
48434,2025-06-01,Uzbekistan,9025,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859
48435,2025-06-01,Uzbekistan,9027,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859
48436,2025-06-01,Uzbekistan,9030,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859


In [4]:
main['logistics_exposure_distw'] = main['gscpi'] * np.log(main['distw'])

#### Дамми по блокам стран

In [5]:
panel_countries = [
    'Albania', 'Armenia', 'Australia', 'Austria', 'Azerbaijan',
    'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Bulgaria',
    'Canada', 'Chile', 'China', 'Croatia', 'Cyprus', 'Czechia',
    'Denmark', 'Ecuador', 'Egypt', 'Estonia', 'Finland', 'France',
    'Georgia', 'Germany', 'Greece', 'Hong Kong', 'Hungary', 'Iceland',
    'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan',
    'Kazakhstan', 'Kyrgyzstan', 'Latvia', 'Lithuania', 'Luxembourg',
    'Malaysia', 'Malta', 'Montenegro', 'Netherlands', 'New Zealand',
    'North Macedonia', 'Northern Ireland', 'Norway', 'Pakistan',
    'Philippines', 'Poland', 'Portugal', 'Rep. of Moldova', 'Romania',
    'Serbia', 'Singapore', 'Slovakia', 'Slovenia', 'South Africa',
    'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Taiwan',
    'Tajikistan', 'Thailand', 'Turkey', 'USA', 'Ukraine',
    'United Kingdom', 'Uzbekistan'
]

# страны ЕС в твоей панели
eu_countries = {
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
    'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
    'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania',
    'Slovakia', 'Slovenia', 'Spain', 'Sweden'
}

# unfriendly по твоему правилу
unfriendly_countries = (
    eu_countries |
    {
        'Australia',
        'Albania',
        'United Kingdom',
        'Iceland',
        'Canada',
        'New Zealand',
        'Norway',
        'South Korea',
        'North Macedonia',
        'Singapore',
        'USA',
        'Taiwan',
        'Ukraine',
        'Montenegro',
        'Switzerland',
        'Japan',
    }
)

# BRICS — из твоего списка панели
brics_countries = {
    'Brazil',
    'China',
    'Egypt',
    'India',
    'Indonesia',
    'South Africa',
}

# 3) EAEU — из твоего списка панели
cis_countries = {
    'Armenia', 'Azerbaijan', 'Kazakhstan', 'Kyrgyzstan',
    'Rep. of Moldova', 'Tajikistan', 'Uzbekistan'
}

# на всякий случай пересечение только с панелью
panel_set = set(panel_countries)
unfriendly_countries = sorted(unfriendly_countries & panel_set)
brics_countries = sorted(brics_countries & panel_set)
cis_countries = sorted(cis_countries & panel_set)

print("unfriendly:", unfriendly_countries)
print("brics:", brics_countries)
print("cis:", cis_countries)

unfriendly: ['Albania', 'Australia', 'Austria', 'Belgium', 'Bulgaria', 'Canada', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Japan', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Montenegro', 'Netherlands', 'New Zealand', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Singapore', 'Slovakia', 'Slovenia', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Taiwan', 'USA', 'Ukraine', 'United Kingdom']
brics: ['Brazil', 'China', 'Egypt', 'India', 'Indonesia', 'South Africa']
cis: ['Armenia', 'Azerbaijan', 'Kazakhstan', 'Kyrgyzstan', 'Rep. of Moldova', 'Tajikistan', 'Uzbekistan']


In [6]:
main['unfriendly'] = main['country'].isin(unfriendly_countries).astype(int)
main['brics'] = main['country'].isin(brics_countries).astype(int)
main['cis'] = main['country'].isin(cis_countries).astype(int)

In [7]:
main['post_sanctions'] = 0
main.loc[main['rep_date'] >= '2022-02-01', 'post_sanctions'] = 1

main['unfriendly_post'] = main['post_sanctions'] * main['unfriendly']
main['brics_post'] = main['post_sanctions'] * main['brics']
main['cis_post'] = main['post_sanctions'] * main['cis']

In [8]:
main

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,distw,logistics_exposure_distw,unfriendly,brics,cis,post_sanctions,unfriendly_post,brics_post,cis_post
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
1,2019-01-01,Albania,9019,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
2,2019-01-01,Albania,9020,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
3,2019-01-01,Albania,9021,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
4,2019-01-01,Albania,9022,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-06-01,Uzbekistan,9022,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859,0.660547,0,0,1,1,0,0,1
48434,2025-06-01,Uzbekistan,9025,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859,0.660547,0,0,1,1,0,0,1
48435,2025-06-01,Uzbekistan,9027,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859,0.660547,0,0,1,1,0,0,1
48436,2025-06-01,Uzbekistan,9030,0.0,0,0,8.494769,7.483273,0.581069,0.084302,2528.859,0.660547,0,0,1,1,0,0,1


In [9]:
main.isna().sum()

rep_date                    0
country                     0
hs                          0
value                       0
sanctions_proxy             0
sanctions_proxy_smooth      0
cpi_yoy                     0
ip_yoy                      0
ex_yoy                      0
gscpi                       0
distw                       0
logistics_exposure_distw    0
unfriendly                  0
brics                       0
cis                         0
post_sanctions              0
unfriendly_post             0
brics_post                  0
cis_post                    0
dtype: int64

In [10]:
main.to_excel('data/final_data.xlsx', index=False)